In [2]:
# pid_lb_demo.py
import os, math, csv, time
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition

# -------------------------
# 基本超参数（尽量“跟你现有设定相近”，但保持最小可复现）
# -------------------------
T = 20            # 离散步数（0..T）
h = 0.2           # 步长
W_E, W_U = 10.0, 0.01   # 目标权重
BOUNDS = {
    "x":(-20,20), "u":(-20,20), "e":(-5,5), "I":(-50,50),
    "Kp":(0,100), "Ki":(0,100), "Kd":(0,100)
}
NODES_TO_RUN = 20        # “跑 20 个节点”
DATA_CSV = "data.csv"    # 读取前 5 个情景
OUT_CSV  = "lb_progress.csv"
OUT_PNG  = "lb_ub_convergence.png"
RANDOM_SEED = 17

# -------------------------
# 读取前 5 个情景
# data.csv 假设包含多行情景，每行含：tau_x,tau_u,tau_d,sp,d_0...d_T
# -------------------------
def load_first5_scenarios(path: str):
    import re, pandas as pd
    df = pd.read_csv(path)

    # 自动推断 T：找 disturbance_数字 列的最大数字
    disturb_cols = [c for c in df.columns if c.startswith("disturbance_")]
    if not disturb_cols:
        raise RuntimeError("找不到 disturbance_* 列。")
    # 取最大下标作为 T（例如 disturbance_0..disturbance_20 -> T=20）
    T = max(int(re.search(r"disturbance_(\d+)$", c).group(1)) for c in disturb_cols)

    scenarios = []
    n = min(5, len(df))
    for i in range(n):
        row = df.iloc[i]
        tau_x = float(row["tau_xs"])           # 注意这里用 tau_xs
        tau_u = float(row["tau_us"])           # 注意这里用 tau_us
        tau_d = float(row["tau_ds"])           # 注意这里用 tau_ds
        sp    = float(row["setpoint_change"])  # 注意这里用 setpoint_change

        d = [ float(row[f"disturbance_{t}"]) for t in range(T+1) ]
        scenarios.append({
            "name": f"scen_{i+1}",
            "tau_x": tau_x, "tau_u": tau_u, "tau_d": tau_d,
            "sp": sp, "d": d, "prob": 1.0/n
        })
    return scenarios, T


# -------------------------
# 构建“共享 Kp,Ki,Kd + 多情景轨迹”的 Pyomo 模型（纯代数形式）
# 动力学离散：后向差分近似
# -------------------------
def build_stochastic_pid_model(scenarios, T, h, bounds, w_e, w_u):
    m = pyo.ConcreteModel()

    # 时间集
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)  # 1..T

    # 一阶段共享变量
    m.Kp = pyo.Var(bounds=bounds["Kp"])
    m.Ki = pyo.Var(bounds=bounds["Ki"])
    m.Kd = pyo.Var(bounds=bounds["Kd"])

    # 为每个情景做一个 Block（便于结构清晰）
    m.S = pyo.Set(initialize=[s["name"] for s in scenarios], ordered=True)
    m.scen = pyo.Block(m.S)

    # 创建变量与约束
    for s in scenarios:
        S = m.scen[s["name"]]
        S.tau_x = pyo.Param(initialize=s["tau_x"])
        S.tau_u = pyo.Param(initialize=s["tau_u"])
        S.tau_d = pyo.Param(initialize=s["tau_d"])
        S.sp    = pyo.Param(initialize=s["sp"])
        # 外扰参数
        S.d = pyo.Param(m.T, initialize=lambda _, t, sdata=s: sdata["d"][t])

        # 轨迹变量
        S.x = pyo.Var(m.T, bounds=bounds["x"])
        S.u = pyo.Var(m.T, bounds=bounds["u"])
        S.e = pyo.Var(m.T, bounds=bounds["e"])
        S.I = pyo.Var(m.T, bounds=bounds["I"])

        # 初值
        S.x0fix = pyo.Constraint(expr=S.x[0] == 0.0)
        S.I0fix = pyo.Constraint(expr=S.I[0] == 0.0)

        # 误差：e = x - sp
        def _e_rule(S, t):
            return S.e[t] == S.x[t] - S.sp
        S.e_con = pyo.Constraint(m.T, rule=_e_rule)

        # 积分：I[t] = I[t-1] + h*e[t]
        def _I_rule(S, t):
            return S.I[t] == S.I[t-1] + h*S.e[t]
        S.I_con = pyo.Constraint(m.Tm, rule=_I_rule)

        # e 的差分（用于 D 项）：de/dt ≈ (e[t]-e[t-1])/h
        # PID：u[t] = Kp*e[t] + Ki*I[t] + Kd*(e[t]-e[t-1])/h
        def _u_rule(S, t):
            if t == 0:
                # 令 D 项在 t=0 为 0（或用 e[-1]=0 近似）
                return S.u[0] == m.Kp*S.e[0] + m.Ki*S.I[0]
            return S.u[t] == m.Kp*S.e[t] + m.Ki*S.I[t] + m.Kd*(S.e[t] - S.e[t-1])/h
        S.u_con = pyo.Constraint(m.T, rule=_u_rule)

        # 系统：x[t] - x[t-1] = h*( -tau_x*x[t] + tau_u*u[t] + tau_d*d[t] )
        def _x_rule(S, t):
            if t == 0:
                return pyo.Constraint.Skip
            return S.x[t] - S.x[t-1] == h*( -S.tau_x*S.x[t] + S.tau_u*S.u[t] + S.tau_d*S.d[t] )
        S.x_con = pyo.Constraint(m.Tm, rule=_x_rule)

        # 情景目标（离散积分）
        S.obj_s = pyo.Expression(expr=sum(h*(w_e*S.e[t]**2 + w_u*S.u[t]**2) for t in m.T))

    # 总目标：期望（等概率）
    def _obj_rule(m):
        total = 0.0
        for s in scenarios:
            S = m.scen[s["name"]]
            total += s["prob"] * S.obj_s
        return total
    m.obj = pyo.Objective(rule=_obj_rule, sense=pyo.minimize)
    return m

# -------------------------
# 主过程：用 Gurobi Persistent 逐次提高 NodeLimit，记录 LB/UB
# -------------------------
def run_lb_progress(m, nodes_to_run=20, time_limit_per_iter=3600, mipgap=0.01):
    try:
        opt = pyo.SolverFactory("gurobi_persistent")
    except Exception as e:
        raise RuntimeError("需要 Gurobi Persistent（gurobipy）与有效 license。") from e

    opt.set_instance(m)
    # 关键选项：全局非凸、日志安静些
    opt.set_gurobi_param("NonConvex", 2)
    opt.set_gurobi_param("OutputFlag", 0)
    # 为每次迭代设置 MIPGap（影响收敛，但我们主要靠 NodeLimit 控制“节点数”）
    opt.set_gurobi_param("MIPGap", mipgap)
    opt.set_gurobi_param("TimeLimit", time_limit_per_iter)

    records = []
    start0 = time.time()
    for k in range(1, nodes_to_run+1):
        # 逐步提升树搜索节点数上限
        opt.set_gurobi_param("NodeLimit", k)

        t0 = time.time()
        res = opt.solve(load_solutions=False)  # 不必每次拉解（LB/UB 用 Gurobi 原生接口）
        t1 = time.time()

        # 通过底层 Gurobi 模型拿 ObjBound (LB) 与 ObjVal (UB)
        gmodel = opt._solver_model
        lb = float(gmodel.ObjBound) if gmodel is not None else float("nan")
        ub = float(gmodel.ObjVal) if gmodel.SolCount > 0 else float("inf")

        gap = abs(ub - lb) / (abs(ub) + 1e-12) if math.isfinite(ub) and math.isfinite(lb) else float("inf")
        status = str(res.solver.status) if hasattr(res, "solver") else "unknown"

        records.append({
            "iter": k,
            "node_limit": k,
            "lb": lb,
            "ub": ub,
            "rel_gap": gap,
            "solve_time_iter": t1 - t0,
            "elapsed": t1 - start0,
            "status": status
        })
        print(f"[{k:02d}] LB={lb:.6g}  UB={ub:.6g}  gap={gap:.3e}  time={t1-t0:.2f}s  status={status}")

    return records

# -------------------------
# 可视化与落盘
# -------------------------
def save_results(records, csv_path=OUT_CSV, png_path=OUT_PNG):
    # CSV
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        w.writeheader()
        for r in records:
            w.writerow(r)

    # Plot
    iters = [r["iter"] for r in records]
    lbs   = [r["lb"] for r in records]
    ubs   = [r["ub"] for r in records]

    plt.figure(figsize=(7,4.2))
    plt.plot(iters, lbs, marker="o", label="Best Bound (LB)")
    plt.plot(iters, ubs, marker="o", label="Incumbent (UB)")
    plt.xlabel("Iteration (NodeLimit)")
    plt.ylabel("Objective")
    plt.title("LB/UB Convergence with Increasing NodeLimit")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(png_path, dpi=160)
    plt.close()

# -------------------------
# main
# -------------------------
def main():
    if not os.path.exists(DATA_CSV):
        raise FileNotFoundError(f"未找到 {DATA_CSV}")

    scenarios, T_inferred = load_first5_scenarios(DATA_CSV)
    # 用自动推断到的 T 覆盖全局 T
    global T
    T = T_inferred

    m = build_stochastic_pid_model(scenarios, T, h, BOUNDS, W_E, W_U)

    # 可选：给 Kp,Ki,Kd 一个初值（帮助 Gurobi 更快找到可行解，从而给出 UB）
    pyo.set_values({
        m.Kp: 1.0,
        m.Ki: 0.5,
        m.Kd: 0.0
    })

    # 逐步提升 NodeLimit，记录 LB/UB
    recs = run_lb_progress(m, nodes_to_run=NODES_TO_RUN, time_limit_per_iter=3600, mipgap=0.01)

    # 把最终可行解提取一下（便于复核轨迹或作图）
    # 注意：上面 solve(load_solutions=False) 没有把解灌回 Pyomo。
    # 这里再做一次轻量 solve 把 incumbent 读回（不影响前面的 LB/UB 日志）
    opt = pyo.SolverFactory("gurobi_persistent")
    opt.set_instance(m)
    opt.set_gurobi_param("NonConvex", 2)
    opt.set_gurobi_param("OutputFlag", 0)
    opt.set_gurobi_param("MIPGap", 0.01)
    opt.set_gurobi_param("TimeLimit", 10)
    opt.solve(load_solutions=True)

    # 保存日志 + 收敛图
    save_results(recs, OUT_CSV, OUT_PNG)
    print(f"\n已保存日志: {OUT_CSV}")
    print(f"已保存收敛图: {OUT_PNG}")

    # 打印一下最优 K（便于 sanity check）
    print(f"Best known K: Kp={pyo.value(m.Kp):.6g}, Ki={pyo.value(m.Ki):.6g}, Kd={pyo.value(m.Kd):.6g}")

if __name__ == "__main__":
    main()


AttributeError: module 'pyomo.environ' has no attribute 'set_values'